In [ ]:
!git clone https://github.com/nbebop/rlhf.git
%cd rlhf

!pip install -q -r requirements.txt

# Colab preinstalls torchao 0.10.0. peft's LoRA path (used once use_lora is
# enabled) probes for torchao and *raises* ImportError on any version below
# 0.16.0 instead of just skipping it -- and this repo never uses torchao at
# all, so the clean fix is to remove it rather than upgrade it (upgrading
# risks pulling a newer torch than Colab's preinstalled CUDA build).
!pip uninstall -y -q torchao

### Mount Google Drive

Mount your Google Drive to save the model outputs. This will ask you to authorize Google Colab to access your Google Drive.

### Link `rlhf/outputs` to Google Drive

This cell will create a symbolic link from the `rlhf/outputs` directory (where the model outputs are saved) to a directory in your Google Drive called `rlhf_outputs`. This ensures all generated models and logs are persistently stored in your Drive.

In [ ]:
import os

# Clear stale mount state before mounting -- on a reused/warm runtime,
# leftover content in /content/drive from a previous session makes
# force_remount=True fail with "Mountpoint must not already contain files".
!fusermount -u /content/drive 2>/dev/null
if os.path.isdir('/content/drive') and not os.path.islink('/content/drive'):
    !rm -rf /content/drive

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

drive_path = '/content/drive/MyDrive/rlhf_outputs'
local_outputs_path = '/content/rlhf/outputs'

# Create the target directory in Google Drive if it doesn't exist
!mkdir -p "{drive_path}"

# Check if the local outputs directory exists and is not a symlink already
if os.path.exists(local_outputs_path) and not os.path.islink(local_outputs_path):
    # If it's a directory, move its contents to the drive_path
    if os.path.isdir(local_outputs_path):
        print(f"Moving contents of {local_outputs_path} to {drive_path}")
        !mv {local_outputs_path}/* "{drive_path}" || true  # Use || true to prevent error if directory is empty
        !rmdir {local_outputs_path}  # Remove the empty local directory
    else:
        print(f"Warning: {local_outputs_path} exists but is not a directory. Skipping content move.")

# Create the symbolic link if it doesn't exist
if not os.path.exists(local_outputs_path):
    print(f"Creating symbolic link from {drive_path} to {local_outputs_path}")
    !ln -s "{drive_path}" {local_outputs_path}
else:
    print(f"Symbolic link or directory {local_outputs_path} already exists. Skipping creation.")

print(f"Content of {local_outputs_path} (now linked to drive): ")
!ls -la {local_outputs_path}

Creating symbolic link from /content/drive/MyDrive/rlhf_outputs to /content/rlhf/outputs
Content of /content/rlhf/outputs (now linked to drive): 
lrwxrwxrwx 1 root root 35 Jul 13 16:27 /content/rlhf/outputs -> /content/drive/MyDrive/rlhf_outputs


In [ ]:
# smoke test
!python src/smoke_test.py --config configs/ckan_config.yaml

=== Stage 1/5: SFT (tiny model, 1 epoch) ===
config.json: 100% 807/807 [00:00<00:00, 3.91MB/s]
tokenizer_config.json: 100% 236/236 [00:00<00:00, 1.64MB/s]
vocab.json: 100% 10.4k/10.4k [00:00<00:00, 26.7MB/s]
merges.txt: 100% 4.88k/4.88k [00:00<00:00, 13.1MB/s]
special_tokens_map.json: 100% 90.0/90.0 [00:00<00:00, 506kB/s]
tokenizer.json: 100% 17.5k/17.5k [00:00<00:00, 9.51MB/s]
model.safetensors: 100% 454k/454k [00:00<00:00, 649kB/s]
Loading weights: 100% 64/64 [00:00<00:00, 9215.09it/s]
Generating train split: 32 examples [00:00, 14293.69 examples/s]
Adding EOS to train dataset: 100% 32/32 [00:00<00:00, 6154.24 examples/s]
Tokenizing train dataset:   0% 0/32 [00:00<?, ? examples/s][transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1228 > 1024). Running this sequence through the model will result in indexing errors
Tokenizing train dataset: 100% 32/32 [00:00<00:00, 555.64 examples/s]
Building labels for train dataset: 100% 

In [ ]:
!python src/01_sft.py --config configs/ckan_config.yaml           # ~5-10 min


config.json: 100% 659/659 [00:00<00:00, 3.72MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 24.3MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 109MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 80.4MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 99.0MB/s]
model.safetensors: 100% 988M/988M [00:07<00:00, 127MB/s]
Loading weights: 100% 290/290 [00:00<00:00, 5496.92it/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.33MB/s]
Generating train split: 600 examples [00:00, 66944.63 examples/s]
Adding EOS to train dataset: 100% 600/600 [00:00<00:00, 15768.26 examples/s]
Tokenizing train dataset:   0% 0/600 [00:00<?, ? examples/s][RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+completion. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+completion. This may be d

In [ ]:
!python src/02_reward_model.py --config configs/ckan_config.yaml  # ~10-15 min

Loading weights: 100% 290/290 [00:00<00:00, 5680.84it/s]
[transformers] Qwen2ForSequenceClassification LOAD REPORT from: /content/rlhf/outputs/ckan_sft_model
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Generating train split: 1200 examples [00:00, 112965.21 examples/s]
Map: 100% 1200/1200 [00:00<00:00, 19866.92 examples/s]
Generating train split: 100 examples [00:00, 45187.50 examples/s]
Map: 100% 100/100 [00:00<00:00, 13614.78 examples/s]
Adding EOS to train dataset: 100% 1200/1200 [00:00<00:00, 21728.30 examples/s]
Tokenizing train dataset: 100% 1200/1200 [00:01<00:00, 797.76 examples/s]
Filtering train >512 tokens: 100% 1200/1200 [00:00<00:00, 4911.72 examples/s]
Adding EOS to eval dataset: 100% 100/100 [00:00<00:00, 14097.08 examples/s]
Tokenizing eval dataset: 100% 100/100 [00:00<00:00, 746.15 examples/s]
Filtering

In [ ]:
# PPO loads 4 copies of the model at once (policy, reference, reward, value)
# and fully trains the value model -- on a T4 (~15 GB usable) that OOMs <even
# at 0.5B params. Enable LoRA on the policy for headroom. Also, batch size 1
# turns total_episodes into 5000 sequential single-sample steps (~16s/it,
# ~22h total) -- try batch 2 (fewer, less serialized steps) and cut episodes
# to a demo-scale run. Drop batch back to 1 if this still OOMs.
# Skip this cell entirely on an L4/A10G/A100 (24 GB+).
import yaml

cfg_path = 'configs/ckan_config.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)
cfg['use_lora'] = True
cfg['ppo']['per_device_train_batch_size'] = 2
cfg['ppo']['total_episodes'] = 500
with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('use_lora:', cfg['use_lora'],
      '| per_device_train_batch_size:', cfg['ppo']['per_device_train_batch_size'],
      '| total_episodes:', cfg['ppo']['total_episodes'])

In [ ]:
!python src/03_ppo_train.py --config configs/ckan_config.yaml     # the long one: ~1-3 h for 5000 episodes


/content/rlhf/src/03_ppo_train.py:23: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.ppo import PPOConfig, PPOTrainer
Loading weights: 100% 290/290 [00:00<00:00, 626.62it/s]
Loading weights: 100% 290/290 [00:00<00:00, 5385.88it/s]
Loading weights: 100% 291/291 [00:00<00:00, 5607.93it/s]
Loading weights: 100% 290/290 [00:00<00:00, 4715.86it/s]
[transformers] Qwen2ForSequenceClassification LOAD REPORT from: /content/rlhf/outputs/ckan_sft_model
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Generating train split: 200 examples [00:00, 95335.92 examples/s]
Map: 100% 200/200 [00:00<00:00, 3439.60 examples/s]
===training policy===
  0% 0/1

In [ ]:
# DPO already fits full fine-tune on a T4 (already verified end-to-end).
# Restore use_lora so it isn't accidentally left on from the PPO cell above.
import yaml

cfg_path = 'configs/ckan_config.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)
cfg['use_lora'] = False
with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('use_lora:', cfg['use_lora'])

In [ ]:
!python src/04_dpo_train.py --config configs/ckan_config.yaml

Loading weights: 100% 290/290 [00:00<00:00, 25290.53it/s]
Adding EOS to train dataset: 100% 1200/1200 [00:00<00:00, 19445.46 examples/s]
Tokenizing train dataset:   0% 0/1200 [00:00<?, ? examples/s][RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rej

In [ ]:
!python src/05_eval_compare.py --config configs/ckan_config.yaml | tee eval_results.txt